# Benchmark against vLLM on an NVIDIA GPU

Runs the full comparison ladder on one workload and one GPU: Hugging Face baseline, our engine (static batching, continuous batching, + paged KV cache, + prefix caching), and **vLLM**.

**Which GPU:** an **L4, A10 or A100** is the best match for the final numbers (Colab: Runtime → Change runtime type → L4 / A100). A **T4** (Kaggle and free Colab) works for our engine in `float16`; recent vLLM releases may not support older GPUs like the T4, and if vLLM fails to start there, the rest of the ladder still runs.

**Kaggle:** Settings → Accelerator → GPU, and turn Internet on.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
!git clone https://github.com/devt-code/llm-serving-engine.git
%cd llm-serving-engine
!pip install -q -r requirements.txt

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen3-0.6B', local_dir='models/qwen3-0.6b')

In [ ]:
import torch
major, _ = torch.cuda.get_device_capability()
DTYPE = 'bfloat16' if major >= 8 else 'float16'
print('using', DTYPE)

## 1. Our engine and the Hugging Face baseline
Run first, before vLLM is installed, so vLLM's dependencies can't change the PyTorch build our numbers use.

In [ ]:
!python bench/ladder.py --dtype $DTYPE --only hf,static,continuous,paged,prefix --out bench/results/ladder_gpu.json

## 2. vLLM
Installed into the same environment after our runs. Same model, same prompts, same output lengths, prefix caching on.

In [ ]:
!pip install -q vllm
!python bench/ladder.py --dtype $DTYPE --only vllm --out bench/results/ladder_gpu.json

In [ ]:
import json
r = json.load(open('bench/results/ladder_gpu.json'))
print(r['gpu'], r['dtype'])
best = max(x['output_tok_per_s'] for x in r['results'] if x['config'] != 'vllm')
for x in r['results']:
    print(f"{x['label']:<36} {x['output_tok_per_s']:8.1f} tok/s")
v = next((x for x in r['results'] if x['config'] == 'vllm'), None)
if v:
    print(f"\nOur best = {100 * best / v['output_tok_per_s']:.0f}% of vLLM's throughput")

In [ ]:
!python -m pytest tests/test_cuda.py -v

Download `bench/results/ladder_gpu.json` (Kaggle: Output tab; Colab: Files panel) and commit it, or paste the printed table back into the chat.